In [6]:
# Import libraries
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import plotly.graph_objs as go
import pandas as pd
import numpy as np
from sklearn.metrics import roc_curve, confusion_matrix, roc_auc_score
from pathlib import Path

# Define paths
DATA_DIR = Path('../Data')

# Load data
y_test = pd.read_csv(DATA_DIR / 'y_test.csv')['default_status']
y_pred_prob = pd.read_csv(DATA_DIR / 'y_pred_prob.csv')['pred_prob']

# Calculate ROC curve and AUC
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)
roc_auc = roc_auc_score(y_test, y_pred_prob)

# Initialize the Dash app
app = dash.Dash(__name__)

# Define the layout of the dashboard
app.layout = html.Div([
    html.H1("Credit Risk Dashboard", style={'textAlign': 'center', 'color': '#1f77b4'}),
    
    # ROC Curve
    html.Div([
        html.H2("ROC Curve"),
        dcc.Graph(id='roc-curve')
    ], style={'margin': '20px'}),
    
    # Predicted Probability Distribution
    html.Div([
        html.H2("Predicted Default Probability Distribution"),
        dcc.Graph(id='probability-distribution')
    ], style={'margin': '20px'}),
    
    # Threshold Slider and Confusion Matrix
    html.Div([
        html.H2("Adjust Classification Threshold"),
        dcc.Slider(
            id='threshold-slider',
            min=0.1,
            max=0.9,
            step=0.1,
            value=0.5,  # Updated to match new threshold
            marks={i/10: str(i/10) for i in range(1, 10)},
            tooltip={"placement": "bottom", "always_visible": True}
        ),
        html.Div([
            html.H3("Confusion Matrix"),
            dcc.Graph(id='confusion-matrix')
        ], style={'margin': '20px'}),
        
        html.Div([
            html.H3("Metrics"),
            html.P(id='tpr-text'),
            html.P(id='fpr-text')
        ], style={'margin': '20px'})
    ], style={'margin': '20px'})
])

# Callback to update ROC Curve (static)
@app.callback(
    Output('roc-curve', 'figure'),
    Input('threshold-slider', 'value')
)
def update_roc_curve(threshold):
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines', name=f'ROC Curve (AUC = {roc_auc:.2f})', line=dict(color='blue')))
    fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', name='Random Guess', line=dict(color='black', dash='dash')))
    fig.update_layout(
        title='Receiver Operating Characteristic (ROC) Curve',
        xaxis_title='False Positive Rate',
        yaxis_title='True Positive Rate',
        legend=dict(x=0.7, y=0.1),
        margin=dict(l=50, r=50, t=50, b=50)
    )
    return fig

# Callback to update Probability Distribution (dynamic based on threshold)
@app.callback(
    Output('probability-distribution', 'figure'),
    Input('threshold-slider', 'value')
)
def update_probability_distribution(threshold):
    below_threshold = y_pred_prob[y_pred_prob < threshold]
    above_threshold = y_pred_prob[y_pred_prob >= threshold]

    fig = go.Figure()
    fig.add_trace(go.Histogram(
        x=below_threshold,
        nbinsx=50,
        name='Predicted No Default',
        marker_color='lightblue',
        opacity=0.7
    ))
    fig.add_trace(go.Histogram(
        x=above_threshold,
        nbinsx=50,
        name='Predicted Default',
        marker_color='orange',
        opacity=0.7
    ))
    fig.add_shape(
        type='line',
        x0=threshold,
        x1=threshold,
        y0=0,
        y1=4500,
        line=dict(color='red', dash='dash'),
        name='Threshold'
    )
    fig.update_layout(
        title='Distribution of Predicted Default Probabilities',
        xaxis_title='Predicted Probability of Default',
        yaxis_title='Frequency',
        barmode='overlay',
        legend=dict(x=0.7, y=0.9),
        margin=dict(l=50, r=50, t=50, b=50)
    )
    return fig

# Callback to update Confusion Matrix and Metrics based on threshold
@app.callback(
    [Output('confusion-matrix', 'figure'),
     Output('tpr-text', 'children'),
     Output('fpr-text', 'children')],
    Input('threshold-slider', 'value')
)
def update_confusion_matrix(threshold):
    y_pred_adjusted = (y_pred_prob >= threshold).astype(int)
    cm = confusion_matrix(y_test, y_pred_adjusted)
    
    # Transpose the confusion matrix to match Seaborn's orientation
    cm = cm.T  # Transpose so that rows are Predicted and columns are Actual

    # Create heatmap for confusion matrix
    fig = go.Figure(data=go.Heatmap(
        z=cm,
        x=['No Default', 'Default'],  # Actual labels (columns after transpose)
        y=['No Default', 'Default'],  # Predicted labels (rows after transpose)
        colorscale='Blues',
        text=cm,
        texttemplate="%{text}",
        textfont={"size": 12},
        showscale=False
    ))
    fig.update_layout(
        title='Confusion Matrix',
        xaxis_title='Actual',
        yaxis_title='Predicted',
        margin=dict(l=50, r=50, t=50, b=50)
    )
    
    # Since we transposed cm, adjust the indices for TN, FP, FN, TP
    tn, fn, fp, tp = cm.ravel()
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    
    tpr_text = f"True Positive Rate (TPR): {tpr:.3f}"
    fpr_text = f"False Positive Rate (FPR): {fpr:.3f}"
    
    return fig, tpr_text, fpr_text

# Run the app
if __name__ == '__main__':
    app.run(debug=True)